# Module 4: SQL-Based Analysis

## Overview

This notebook generates business insights from the cleaned retail transaction dataset using SQL queries.

It produces analytical reports such as Monthly Revenue Analysis, Top Customers, Payment Method Distribution, and Product Category Performance. These reports help organizations understand sales trends and customer behavior.

In [0]:
# Load Final/Clean Layer table

clean_df = spark.table("trustguard.clean_transactions")

# Create temporary SQL view

clean_df.createOrReplaceTempView("transactions")

print("SQL view created successfully.")

SQL view created successfully.


In [0]:
%sql

-- Top 10 Monthly Revenue by Location

SELECT
    location,
    DATE_FORMAT(transaction_date, 'yyyy-MM') AS month,
    ROUND(SUM(total_spent), 2) AS total_revenue
FROM transactions
GROUP BY
    location,
    DATE_FORMAT(transaction_date, 'yyyy-MM')
ORDER BY
    total_revenue DESC
LIMIT 10;

location,month,total_revenue
Online,2022-01,28930.0
In-store,2023-01,28684.5
Online,2022-09,27784.0
In-store,2024-01,27691.0
In-store,2024-12,27303.0
Online,2024-04,27007.0
In-store,2022-01,26781.5
Online,2024-11,26383.0
In-store,2024-05,25461.0
Online,2023-07,24965.0


In [0]:
%sql

-- Find Top 10 Customers by Total Spend

SELECT
    customer_id,
    SUM(total_spent) AS total_spent

FROM transactions

GROUP BY customer_id

ORDER BY total_spent DESC

LIMIT 10;

customer_id,total_spent
CUST_24,71317.0
CUST_08,70244.0
CUST_05,70022.0
CUST_13,68259.5
CUST_23,67879.5
CUST_16,67783.0
CUST_10,65875.5
CUST_22,65272.5
CUST_15,65187.5
CUST_21,64960.5


In [0]:
%sql

-- Calculate percentage share of each payment method

SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM transactions),
        2
    ) AS percentage_share

FROM transactions

GROUP BY payment_method

ORDER BY percentage_share DESC;

payment_method,total_transactions,percentage_share
Cash,4310,34.27
Digital Wallet,4144,32.95
Credit Card,4121,32.77


In [0]:
%sql

-- Calculate revenue generated by each product category

SELECT
    category,
    COUNT(*) AS total_products,
    SUM(total_spent) AS total_revenue

FROM transactions

GROUP BY category

ORDER BY total_revenue DESC;

category,total_products,total_revenue
Butchers,1568,216480.5
Electric Household Essentials,1591,213383.5
Beverages,1567,205220.0
Furniture,1591,203710.0
Food,1588,203489.5
Computers And Electric Accessories,1558,200135.0
Patisserie,1528,192470.5
Milk Products,1584,188262.0


In [0]:
# Save all SQL analysis results as Delta tables

# Save Monthly Revenue

spark.sql("""
SELECT
    location,
    date_format(transaction_date,'yyyy-MM') AS month,
    SUM(total_spent) AS total_revenue

FROM transactions

GROUP BY
    location,
    date_format(transaction_date,'yyyy-MM')

ORDER BY
    month,
    location

""").write.mode("overwrite").saveAsTable("trustguard.monthly_revenue")


# Top Customers
spark.sql("""
SELECT
    customer_id,
    SUM(total_spent) AS total_spent
FROM transactions
GROUP BY customer_id
ORDER BY total_spent DESC
LIMIT 10
""").write.mode("overwrite").saveAsTable("trustguard.top_customers")


# Payment Method Share
spark.sql("""
SELECT
    payment_method,
    COUNT(*) AS total_transactions,
    ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM transactions),2) AS percentage_share
FROM transactions
GROUP BY payment_method
""").write.mode("overwrite").saveAsTable("trustguard.payment_method_share")


# Category Breakdown
spark.sql("""
SELECT
    category,
    COUNT(*) AS total_products,
    SUM(total_spent) AS total_revenue
FROM transactions
GROUP BY category
""").write.mode("overwrite").saveAsTable("trustguard.category_breakdown")

print("Analysis tables saved successfully.")

Analysis tables saved successfully.


In [0]:
# Verify all saved analysis tables

print("===== Monthly Revenue =====")

display(
    spark.sql("""
    SELECT *
    FROM trustguard.monthly_revenue
    ORDER BY month, location
    LIMIT 10
    """)
)

print("===== Top 10 Customers =====")
display(
    spark.table("trustguard.top_customers")
)

print("===== Payment Method Share =====")
display(
    spark.table("trustguard.payment_method_share")
)

print("===== Category Breakdown =====")
display(
    spark.table("trustguard.category_breakdown")
)

===== Monthly Revenue =====


location,month,total_revenue
In-store,2022-01,26781.5
Online,2022-01,28930.0
In-store,2022-02,23061.0
Online,2022-02,22717.0
In-store,2022-03,22045.5
Online,2022-03,20808.0
In-store,2022-04,19543.5
Online,2022-04,22683.5
In-store,2022-05,18768.5
Online,2022-05,23011.5


===== Top 10 Customers =====


customer_id,total_spent
CUST_24,71317.0
CUST_08,70244.0
CUST_05,70022.0
CUST_13,68259.5
CUST_23,67879.5
CUST_16,67783.0
CUST_10,65875.5
CUST_22,65272.5
CUST_15,65187.5
CUST_21,64960.5


===== Payment Method Share =====


payment_method,total_transactions,percentage_share
Digital Wallet,4144,32.95
Credit Card,4121,32.77
Cash,4310,34.27


===== Category Breakdown =====


category,total_products,total_revenue
Computers And Electric Accessories,1558,200135.0
Beverages,1567,205220.0
Electric Household Essentials,1591,213383.5
Butchers,1568,216480.5
Food,1588,203489.5
Milk Products,1584,188262.0
Patisserie,1528,192470.5
Furniture,1591,203710.0
